In [31]:
!rm -rf dataset/ dataset.zip

!wget -nc -q -O "./dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/oddrationale/mnist-in-csv"
!unzip -q dataset.zip

In [32]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [33]:
# Upload dataset zip file to S3
from sagemaker.s3 import S3Uploader

inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [37]:
import pandas as pd

df = pd.read_csv('./mnist_train.csv')
print(df.shape)
print(df.isnull().sum())
print(df.columns)
df.head()

(60000, 785)
label    0
1x1      0
1x2      0
1x3      0
1x4      0
        ..
28x24    0
28x25    0
28x26    0
28x27    0
28x28    0
Length: 785, dtype: int64
Index(['label', '1x1', '1x2', '1x3', '1x4', '1x5', '1x6', '1x7', '1x8', '1x9',
       ...
       '28x19', '28x20', '28x21', '28x22', '28x23', '28x24', '28x25', '28x26',
       '28x27', '28x28'],
      dtype='object', length=785)


,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [51]:
from sagemaker.xgboost import XGBoost

estimator = XGBoost(
    entry_point="train.py",
    source_dir="src",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="3.0-5",
    hyperparameters={
      "n_estimators": 100,
      "max_depth": 5,
      "learning_rate": 0.05,
    },
)

estimator.fit(inputs={"train": f's3://{bucket}/{prefix}/dataset.zip'}, wait=True)

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-06-30-07-17-46-513


2026-06-30 07:17:48 Starting - Starting the training job...
2026-06-30 07:18:03 Starting - Preparing the instances for training...
2026-06-30 07:18:24 Downloading - Downloading input data...
2026-06-30 07:19:15 Downloading - Downloading the training image......
2026-06-30 07:20:05 Training - Training image download completed. Training in progress./miniconda3/lib/python3.10/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-30:07:20:11:INFO] Imported framework sagemaker_xgboost_container.training
[2026-06-30:07:20:11:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-30:07:20:11:INFO] Invoking user training script.
[2026-06-30:07:20:11:INFO] Module train does not provide a setup.py. 
Generating setup.py
[202

In [52]:
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model()
model_name = f'mnist-model-{int(time.time())}'
model._create_sagemaker_model(instance_type="ml.m5.large")

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-06-30-07-27-15-500


In [54]:
new_config = f'mnist-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": "sagemaker-xgboost-2026-06-30-07-27-15-500",
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.large"
    }]
)

sm.update_endpoint(
    EndpointName = "mnist-endpoint",
    EndpointConfigName = new_config
)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/mnist-endpoint',
 'ResponseMetadata': {'RequestId': 'b4018812-f10e-4374-880a-e77cb4c28dd0',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'b4018812-f10e-4374-880a-e77cb4c28dd0',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '82',
   'date': 'Tue, 30 Jun 2026 07:28:08 GMT'},
  'RetryAttempts': 0}}

In [57]:
import json, boto3, base64

client = boto3.client("sagemaker-runtime")
endpoint_name = 'mnist-endpoint'

with open("9963.jpg", "rb") as image_file:
    data = base64.b64encode(image_file.read()).decode('utf-8')

body = {"image": data}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(body),
)
result = json.load(response["Body"])
print(result)
print(result['class_name'][0])


{'class_name': [7]}
7
